# Техническое задание: дашборд аналитики League of Legends

Автор: Стукалов Артем Витальевич  
Дата: 05.07.2026

## Цель работы

Разработать интерактивный дашборд, отображающий статистику матчей, игроков и чемпионов Legue of Legends за текущий месяц для регионов Европа и США. Дашборд должен быть реализован на Python с использованием библиотек pandas, plotly, dash и официального API Riot Games.

## 1. Установка и импорт библиотек

In [1]:
import requests   # для HTTP-запросов к API
import pandas as pd  # для работы с таблицами
import time       # чтобы делать паузы между запросами и не получить бан от API
import os # для работы с файловой системой (пути к файлам)
from dotenv import load_dotenv # для загрузки переменных окружения из файла .env
from sqlalchemy import create_engine 

## 2. Настройка API-ключа и серверов

In [2]:
# Загружаем переменные из файла riot_key.env
load_dotenv("riot_key.env")

# Читаем API-ключ из переменной окружения
RIOT_API_KEY = os.getenv("RIOT_API_KEY")

# Серверы
REGION       = "euw1"    # для запросов к лигам и игрокам
MATCH_REGION = "europe"  # для запросов к матчам

# Headers — это служебная информация, которую мы отправляем вместе с каждым запросом.
# Riot API проверяет наш ключ именно через заголовок X-Riot-Token.
headers = {
    "X-Riot-Token": RIOT_API_KEY
}

## 3. Получаем список топовых игроков (Challenger)

Мы начнём с **Challenger** - это топ сервера по рейтингу. Riot предоставляет полный список таких игроков через один запрос.

In [3]:
# Формируем URL запроса к Challenger-лиге
url = (
    f"https://{REGION}.api.riotgames.com"
    f"/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
)

# Отправляем GET-запрос
response = requests.get(url, headers=headers)

# Смотрим статус ответа
# 200 — успех, 401/403 — проблема с ключом, 429 — превышен лимит запросов
print("Статус ответа:", response.status_code)

Статус ответа: 200


In [4]:
# Превращаем JSON-ответ в Python-словарь
data = response.json()

# Смотрим, какие ключи есть в ответе
data.keys()

dict_keys(['tier', 'queue', 'entries'])

In [5]:
# Полезная информация о лиге
print("Тир:", data["tier"])
print("Очередь:", data["queue"])
print("Количество игроков:", len(data["entries"]))

Тир: CHALLENGER
Очередь: RANKED_SOLO_5x5
Количество игроков: 310


In [6]:
# 'entries' - список игроков в этой лиге
# Каждый элемент - словарь с данными об одном игроке
players = data["entries"]

print("Количество игроков:", len(players))
print()
print("Пример — первый игрок:")
players[0]

Количество игроков: 310

Пример — первый игрок:


{'puuid': 'ssoDQr8vZFdQX5nvMUWQ53tEB6sb7Gooxypsymi6MbtE4T1UJ7SQTHACs3CDULdCn2i_wqCyn-qjag',
 'leaguePoints': 4028,
 'rank': 'I',
 'wins': 435,
 'losses': 288,
 'veteran': True,
 'inactive': False,
 'freshBlood': False,
 'hotStreak': False}

### Превращаем список игроков в таблицу

In [7]:
# Создаём DataFrame прямо из списка игроков
players_df = pd.DataFrame(players)

# Сортируем по очкам LP (leaguePoints) — от большего к меньшему
players_df = players_df.sort_values("leaguePoints", ascending=False).reset_index(drop=True)

print("Размер таблицы:", players_df.shape)
players_df.head()

Размер таблицы: (310, 9)


,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak
0,ssoDQr8vZFdQX5nvMUWQ53tEB6sb7Gooxypsymi6MbtE4T...,4028,I,435,288,True,False,False,False
1,K4G9h9Fo43Ft65sQJTiY929UMRJ2Er_ydlmmSdqEt59dlS...,4010,I,809,676,True,False,False,True
2,Xbjv6OB74DFVLYUf7X3xDe7kKdrYlT7Bgsdj8XO5RQPlFv...,3961,I,765,643,True,False,False,False
3,PBel_iC0Hyn0vl5G2Hu15xn4C4uBlLCnii21SZ4jJr-WUE...,3900,I,521,444,True,False,False,False
4,bTl6NDwSnvh7TjCxFHjQdovyN8ddSYUZPpuLUWteMN5Al5...,3878,I,1356,1145,True,False,False,True


In [8]:
# Сохраняем таблицу игроков в CSV
players_df.to_csv("players.csv", index=False)
print("Сохранено: players.csv")

Сохранено: players.csv


### **Задание 1**
Посмотрите на таблицу `players_df`.
- Сколько игроков имеют больше 500 побед (`wins > 500`)?
- Выведите топ-5 игроков по количеству LP.

Выводим игроков, которые имеют больше 500 побед

In [9]:
# Фильтруем игроков по количеству побед
count_500_wins = players_df[players_df["wins"] > 500].shape[0]
print(f"Количество игроков с более 500 побед: {count_500_wins}")

Количество игроков с более 500 побед: 106


Считаем топ‑5 игроков по количеству LP (leaguePoints)

In [10]:
# Топ‑5 игроков по количеству LP
top5_lp = players_df.sort_values("leaguePoints", ascending=False).head(5)[["puuid", "leaguePoints", "wins", "losses"]]
print("\nТоп‑5 по LP:")
print(top5_lp)


Топ‑5 по LP:
                                               puuid  leaguePoints  wins  \
0  ssoDQr8vZFdQX5nvMUWQ53tEB6sb7Gooxypsymi6MbtE4T...          4028   435   
1  K4G9h9Fo43Ft65sQJTiY929UMRJ2Er_ydlmmSdqEt59dlS...          4010   809   
2  Xbjv6OB74DFVLYUf7X3xDe7kKdrYlT7Bgsdj8XO5RQPlFv...          3961   765   
3  PBel_iC0Hyn0vl5G2Hu15xn4C4uBlLCnii21SZ4jJr-WUE...          3900   521   
4  bTl6NDwSnvh7TjCxFHjQdovyN8ddSYUZPpuLUWteMN5Al5...          3878  1356   

   losses  
0     288  
1     676  
2     643  
3     444  
4    1145  


## 4. Получаем матчи игроков

Match API работает по цепочке:
1. Берём `puuid` игрока из таблицы players
2. Запрашиваем список ID матчей для этого игрока
3. По каждому ID скачиваем подробные данные матча
4. Из каждого матча извлекаем статистику 10 участников

In [11]:
# Берём топ-5 игроков для сбора матчей
TOP_PLAYERS   = 5   # сколько игроков обрабатывать
MATCHES_COUNT = 5   # сколько матчей брать на каждого игрока

all_matches = []  # сюда будем складывать строки для итоговой таблицы

for i, player in players_df.head(TOP_PLAYERS).iterrows():

    puuid = player["puuid"]
    print(f"Игрок {i+1}: {player['summonerName'] if 'summonerName' in player else puuid[:16]}...")

    # Шаг 1: получаем список ID матчей для этого игрока
    ids_url = (
        f"https://{MATCH_REGION}.api.riotgames.com"
        f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
        f"?start=0&count={MATCHES_COUNT}"
    )
    resp = requests.get(ids_url, headers=headers)

    if resp.status_code != 200:
        print(f"Ошибка при получении матчей: {resp.status_code}")
        continue

    match_ids = resp.json()  # список строк вида "EUW1_7123456789"
    print(f"  Найдено матчей: {len(match_ids)}")

    # Шаг 2: скачиваем подробные данные каждого матча
    for match_id in match_ids:

        match_url = (
            f"https://{MATCH_REGION}.api.riotgames.com"
            f"/lol/match/v5/matches/{match_id}"
        )
        match_resp = requests.get(match_url, headers=headers)

        if match_resp.status_code != 200:
            print(f"Ошибка матча {match_id}: {match_resp.status_code}")
            continue

        match_json = match_resp.json()

        # В ответе два раздела: 'metadata' и 'info'
        # Нам нужен 'info' - там вся игровая информация
        game_info     = match_json["info"]
        game_duration = game_info["gameDuration"]  # длительность в секундах
        participants  = game_info["participants"]   # список 10 игроков

        # Шаг 3: определяем победившую команду
        # info["teams"] — список из двух элементов (синяя и красная команда)
        # Каждый элемент: {"teamId": 100/200, "win": True/False, ...}
        # teamId 100 = Blue (синяя сторона), 200 = Red (красная сторона)
        #
        # Мы строим словарь: teamId -> название команды
        # Это позволит нам потом быстро найти команду любого участника
        team_id_to_name = {100: "Blue", 200: "Red"}

        # Узнаём, какой teamId победил
        winning_team_id = None
        for team in game_info["teams"]:
            if team["win"]:
                winning_team_id = team["teamId"]  # 100 или 200
                break

        # Переводим числовой ID в читаемое название: 100 -> "Blue", 200 -> "Red"
        winning_team_name = team_id_to_name.get(winning_team_id, "Unknown")

        # Шаг 4: извлекаем нужные поля по каждому участнику
        for p in participants:
            # У каждого участника есть teamId — 100 (Blue) или 200 (Red)
            player_team_id   = p.get("teamId")
            player_team_name = team_id_to_name.get(player_team_id, "Unknown")

            row = {
                "match_id":      match_id,
                "puuid":         p.get("puuid"),
                "champion":      p.get("championName"),
                "team":          player_team_name,   # Blue или Red — команда игрока
                "winning_team":  winning_team_name,  # Blue или Red — победившая команда
                "kills":         p.get("kills"),
                "deaths":        p.get("deaths"),
                "assists":       p.get("assists"),
                "gold_earned":   p.get("goldEarned"),
                "damage_to_champions": p.get("totalDamageDealtToChampions"),
                "minions_killed":      p.get("totalMinionsKilled"),
                "vision_score":        p.get("visionScore"),
                "win":           p.get("win"),        # True/False для конкретного игрока
                "role":          p.get("role"),
                "team_position": p.get("teamPosition"),
                "game_duration_sec": game_duration,
            }
            all_matches.append(row)

        # Пауза между запросами — Riot ограничивает: ~20 запросов в секунду
        # для бесплатного ключа. Пауза в 1 секунду — надёжная защита от бана.
        time.sleep(1.2)

print(f"\nВсего собрано строк: {len(all_matches)}")

Игрок 1: ssoDQr8vZFdQX5nv...
  Найдено матчей: 5
Игрок 2: K4G9h9Fo43Ft65sQ...
  Найдено матчей: 5
Игрок 3: Xbjv6OB74DFVLYUf...
  Найдено матчей: 5
Игрок 4: PBel_iC0Hyn0vl5G...
  Найдено матчей: 5
Игрок 5: bTl6NDwSnvh7TjCx...
  Найдено матчей: 5

Всего собрано строк: 250


In [12]:
# Создаём DataFrame из всех собранных строк
matches_df = pd.DataFrame(all_matches)

print("Размер таблицы:", matches_df.shape)
matches_df.head()

Размер таблицы: (250, 16)


,match_id,puuid,champion,team,winning_team,kills,deaths,assists,gold_earned,damage_to_champions,minions_killed,vision_score,win,role,team_position,game_duration_sec
0,EUW1_7903341045,4at6aYMfmRjhILQuNy_7vLOpKMWcl-8Sqq3qJ7lEC26OkS...,Singed,Blue,Red,0,2,5,7742,15291,204,18,False,NONE,TOP,1466
1,EUW1_7903341045,tN5_imed5Exq8ky2aOPLs9LOqDrWeM_sRwAelmFb19S03Z...,Elise,Blue,Red,2,7,7,8227,19059,6,22,False,NONE,JUNGLE,1466
2,EUW1_7903341045,1W-qsRJIfO3JVrKd4OyIjdcX4PUodkAsl8pXYhbfxOBaV-...,Yasuo,Blue,Red,2,3,1,7868,10858,171,14,False,SOLO,MIDDLE,1466
3,EUW1_7903341045,4gg2vM-4u8wfAwpw2ot3zDqsnRBrDixvGB91vnooJe9ytv...,Varus,Blue,Red,10,7,1,12781,22637,232,16,False,CARRY,BOTTOM,1466
4,EUW1_7903341045,_e7dWtv-BUliXInflL39nVW5GhdZsLd61KeqTDK87OLfKs...,Lulu,Blue,Red,1,2,13,7074,9932,26,103,False,SUPPORT,UTILITY,1466


In [13]:
# Смотрим типы данных и пропуски
matches_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   match_id             250 non-null    object
 1   puuid                250 non-null    object
 2   champion             250 non-null    object
 3   team                 250 non-null    object
 4   winning_team         250 non-null    object
 5   kills                250 non-null    int64 
 6   deaths               250 non-null    int64 
 7   assists              250 non-null    int64 
 8   gold_earned          250 non-null    int64 
 9   damage_to_champions  250 non-null    int64 
 10  minions_killed       250 non-null    int64 
 11  vision_score         250 non-null    int64 
 12  win                  250 non-null    bool  
 13  role                 250 non-null    object
 14  team_position        250 non-null    object
 15  game_duration_sec    250 non-null    int64 
dtypes: bool(

In [14]:
# Сохраняем таблицу матчей в CSV
matches_df.to_csv("matches.csv", index=False)
print("Сохранено: matches.csv")

Сохранено: matches.csv


### **Задание 2**
Посмотрите на таблицу `matches_df`.
- Какой чемпион встречается чаще всего?
- Переведите `game_duration_sec` из секунд в минуты и добавьте колонку `game_duration_min`.
- Посчитайте среднюю длительность матча в минутах.

 Выделяем игрока, который чаще всего становился чемпионов

In [34]:
# Выделяем игрока, который чаще всего становился чемпионов
champion_counts = matches_df["champion"].value_counts()
most_common = champion_counts.sort_values(ascending=False).index[0]
most_common_count = champion_counts.max()
print(f"Самый частый чемпион: {most_common} (встречается {most_common_count} раз)")
print("\nТоп-15 чемпионов по частоте:")
print(champion_counts.head(15))

Самый частый чемпион: Nautilus (встречается 8 раз)

Топ-15 чемпионов по частоте:
champion
Nautilus     8
Ahri         7
Ezreal       6
Seraphine    5
Ekko         5
Yasuo        5
Akshan       5
Sylas        5
Jhin         5
Draven       4
Camille      4
Elise        4
Renekton     4
Yone         4
Ziggs        4
Name: count, dtype: int64


Переводим game_duration_sec из секунд в минуты и добавляем колонку game_duration_min.

In [16]:
# Добавляем колонку с длительностью в минутах
matches_df["game_duration_min"] = matches_df["game_duration_sec"] / 60

# Проверяем результат
matches_df.head()

,match_id,puuid,champion,team,winning_team,kills,deaths,assists,gold_earned,damage_to_champions,minions_killed,vision_score,win,role,team_position,game_duration_sec,game_duration_min
0,EUW1_7903341045,4at6aYMfmRjhILQuNy_7vLOpKMWcl-8Sqq3qJ7lEC26OkS...,Singed,Blue,Red,0,2,5,7742,15291,204,18,False,NONE,TOP,1466,24.433333
1,EUW1_7903341045,tN5_imed5Exq8ky2aOPLs9LOqDrWeM_sRwAelmFb19S03Z...,Elise,Blue,Red,2,7,7,8227,19059,6,22,False,NONE,JUNGLE,1466,24.433333
2,EUW1_7903341045,1W-qsRJIfO3JVrKd4OyIjdcX4PUodkAsl8pXYhbfxOBaV-...,Yasuo,Blue,Red,2,3,1,7868,10858,171,14,False,SOLO,MIDDLE,1466,24.433333
3,EUW1_7903341045,4gg2vM-4u8wfAwpw2ot3zDqsnRBrDixvGB91vnooJe9ytv...,Varus,Blue,Red,10,7,1,12781,22637,232,16,False,CARRY,BOTTOM,1466,24.433333
4,EUW1_7903341045,_e7dWtv-BUliXInflL39nVW5GhdZsLd61KeqTDK87OLfKs...,Lulu,Blue,Red,1,2,13,7074,9932,26,103,False,SUPPORT,UTILITY,1466,24.433333


Считаем среднюю длительность матча в минутах.

In [17]:
# Добавляем столбец со средней длительностью матча в минутах
avg_duration = matches_df["game_duration_min"].mean()
print(f"\nСредняя длительность матча: {avg_duration:.2f} минут")


Средняя длительность матча: 27.65 минут


## 5. Получаем справочник чемпионов

Riot предоставляет **Data Dragon** — статический репозиторий с данными об игре: чемпионы, предметы, руны.


In [18]:
# Запрос к Data Dragon — ключ не нужен
url = "https://ddragon.leagueoflegends.com/cdn/14.10.1/data/en_US/champion.json"

response = requests.get(url)
print("Статус:", response.status_code)

champions_data = response.json()

Статус: 200


In [19]:
# Смотрим верхний уровень JSON
champions_data.keys()

dict_keys(['type', 'format', 'version', 'data'])

In [20]:
# В ключе 'data' лежит словарь: имя_чемпиона -> его данные
champions = champions_data["data"]

print("Всего чемпионов:", len(champions))
print()

# Пример: данные одного чемпиона
champions["Alistar"]

Всего чемпионов: 167



{'version': '14.10.1',
 'id': 'Alistar',
 'key': '12',
 'name': 'Alistar',
 'title': 'the Minotaur',
 'blurb': 'Always a mighty warrior with a fearsome reputation, Alistar seeks revenge for the death of his clan at the hands of the Noxian empire. Though he was enslaved and forced into the life of a gladiator, his unbreakable will was what kept him from truly...',
 'info': {'attack': 6, 'defense': 9, 'magic': 5, 'difficulty': 7},
 'image': {'full': 'Alistar.png',
  'sprite': 'champion0.png',
  'group': 'champion',
  'x': 192,
  'y': 0,
  'w': 48,
  'h': 48},
 'tags': ['Tank', 'Support'],
 'partype': 'Mana',
 'stats': {'hp': 685,
  'hpperlevel': 120,
  'mp': 350,
  'mpperlevel': 40,
  'movespeed': 330,
  'armor': 47,
  'armorperlevel': 4.7,
  'spellblock': 32,
  'spellblockperlevel': 2.05,
  'attackrange': 125,
  'hpregen': 8.5,
  'hpregenperlevel': 0.85,
  'mpregen': 8.5,
  'mpregenperlevel': 0.8,
  'crit': 0,
  'critperlevel': 0,
  'attackdamage': 62,
  'attackdamageperlevel': 3.75,
  

In [21]:
# Проходим по всем чемпионам и собираем нужные поля
champions_rows = []

for champ_name, champ_info in champions.items():
    row = {
        "champion_id":   champ_info["key"],              # числовой ID (строка)
        "champion_name": champ_name,                     # имя (ключ в словаре)
        "title":         champ_info["title"],             # заголовок, напр. "the Minotaur"
        "tags":          ",".join(champ_info["tags"]),    # теги: Tank, Fighter, Mage и т.д.
    }
    champions_rows.append(row)

# Создаём DataFrame
champions_df = pd.DataFrame(champions_rows)

print("Размер таблицы:", champions_df.shape)
champions_df.sample(5)

Размер таблицы: (167, 4)


,champion_id,champion_name,title,tags
102,33,Rammus,the Armordillo,"Tank,Fighter"
15,63,Brand,the Burning Vengeance,Mage
58,55,Katarina,the Sinister Blade,"Assassin,Mage"
52,222,Jinx,the Loose Cannon,Marksman
80,21,MissFortune,the Bounty Hunter,Marksman


In [22]:
# Сохраняем справочник чемпионов
champions_df.to_csv("champions.csv", index=False)
print("Сохранено: champions.csv")

Сохранено: champions.csv


### **Задание 3**
Работаем со справочником чемпионов.
- Сколько чемпионов каждого типа (`tags`)?
- Попробуйте объединить `matches_df` и `champions_df` по полю `champion` / `champion_name`, используя `pd.merge()`. Что получается?

In [23]:
# Разделяем строку с тегами, превращаем в столбец и считаем частоту
all_tags = champions_df["tags"].str.split(",").explode()
tag_counts = all_tags.value_counts()
print("Количество чемпионов по каждому тегу:")
print(tag_counts)

Количество чемпионов по каждому тегу:
tags
Fighter     71
Mage        63
Tank        45
Assassin    43
Support     36
Marksman    31
Name: count, dtype: int64


Объединим датасеты matches_df и champions_df по полю champion / champion_name

In [24]:
# Выводим основную информацию matches_df
matches_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   match_id             250 non-null    object 
 1   puuid                250 non-null    object 
 2   champion             250 non-null    object 
 3   team                 250 non-null    object 
 4   winning_team         250 non-null    object 
 5   kills                250 non-null    int64  
 6   deaths               250 non-null    int64  
 7   assists              250 non-null    int64  
 8   gold_earned          250 non-null    int64  
 9   damage_to_champions  250 non-null    int64  
 10  minions_killed       250 non-null    int64  
 11  vision_score         250 non-null    int64  
 12  win                  250 non-null    bool   
 13  role                 250 non-null    object 
 14  team_position        250 non-null    object 
 15  game_duration_sec    250 non-null    int

In [25]:
# Выводим основную информацию champions_df
champions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   champion_id    167 non-null    object
 1   champion_name  167 non-null    object
 2   title          167 non-null    object
 3   tags           167 non-null    object
dtypes: object(4)
memory usage: 5.3+ KB


In [26]:
# Объединеняем таблицы матчей
merged_df = pd.merge(
    matches_df,
    champions_df,
    left_on="champion",
    right_on="champion_name",
    how="left"
)

print("\nРазмер объединённой таблицы:", merged_df.shape)
print("\nПервые 5 строк объединённой таблицы:")
print(merged_df.head())

# Проверяем: есть ли чемпионы из матчей
missing = merged_df[merged_df["champion_name"].isna()]
print(f"\nКоличество записей матчей с отсутствующим чемпионом в справочнике: {len(missing)}")
if len(missing) > 0:
    print("Примеры отсутствующих имён:", missing["champion"].unique()[:5])


Размер объединённой таблицы: (250, 21)

Первые 5 строк объединённой таблицы:
          match_id                                              puuid  \
0  EUW1_7903341045  4at6aYMfmRjhILQuNy_7vLOpKMWcl-8Sqq3qJ7lEC26OkS...   
1  EUW1_7903341045  tN5_imed5Exq8ky2aOPLs9LOqDrWeM_sRwAelmFb19S03Z...   
2  EUW1_7903341045  1W-qsRJIfO3JVrKd4OyIjdcX4PUodkAsl8pXYhbfxOBaV-...   
3  EUW1_7903341045  4gg2vM-4u8wfAwpw2ot3zDqsnRBrDixvGB91vnooJe9ytv...   
4  EUW1_7903341045  _e7dWtv-BUliXInflL39nVW5GhdZsLd61KeqTDK87OLfKs...   

  champion  team winning_team  kills  deaths  assists  gold_earned  \
0   Singed  Blue          Red      0       2        5         7742   
1    Elise  Blue          Red      2       7        7         8227   
2    Yasuo  Blue          Red      2       3        1         7868   
3    Varus  Blue          Red     10       7        1        12781   
4     Lulu  Blue          Red      1       2       13         7074   

   damage_to_champions  ...  vision_score    win     role team

## 6. Итог
Мы прошли полный ETL-цикл:  
- **Extract** - запросили Challenger-лигу, матчи, справочник чемпионов через API
- **Transform** - разобрали JSON, выбрали нужные поля, собрали списки словарей
- **Load** - создали DataFrames и сохранили в CSV

**Получившиеся файлы:**
- `players.csv` — топовые игроки сервера
- `matches.csv` — матчи с детальной статистикой участников
- `champions.csv` — справочник всех чемпионов

Объединим для удобства дальнейшего анализа полученные три таблицы

In [27]:
# Выводим основную информацию о датасете players_df
players_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   puuid         310 non-null    object
 1   leaguePoints  310 non-null    int64 
 2   rank          310 non-null    object
 3   wins          310 non-null    int64 
 4   losses        310 non-null    int64 
 5   veteran       310 non-null    bool  
 6   inactive      310 non-null    bool  
 7   freshBlood    310 non-null    bool  
 8   hotStreak     310 non-null    bool  
dtypes: bool(4), int64(3), object(2)
memory usage: 13.4+ KB


In [28]:
# Создаём общий датасет по общему ключу puuid
final_df = pd.merge(
    merged_df,
    players_df,
    on="puuid",
    how="left"
)

print("Размер итоговой таблицы:", final_df.shape)
print("\nДобавились столбцы из players_df:", 
      [col for col in players_df.columns if col not in merged_df.columns])
print("\nПервые 5 строк датасета final_df:")

# Показывать все колонки без обрезания
pd.set_option('display.max_columns', None)

# Проверяем итоговый датасет
print(final_df.head())

Размер итоговой таблицы: (250, 29)

Добавились столбцы из players_df: ['leaguePoints', 'rank', 'wins', 'losses', 'veteran', 'inactive', 'freshBlood', 'hotStreak']

Первые 5 строк датасета final_df:
          match_id                                              puuid  \
0  EUW1_7903341045  4at6aYMfmRjhILQuNy_7vLOpKMWcl-8Sqq3qJ7lEC26OkS...   
1  EUW1_7903341045  tN5_imed5Exq8ky2aOPLs9LOqDrWeM_sRwAelmFb19S03Z...   
2  EUW1_7903341045  1W-qsRJIfO3JVrKd4OyIjdcX4PUodkAsl8pXYhbfxOBaV-...   
3  EUW1_7903341045  4gg2vM-4u8wfAwpw2ot3zDqsnRBrDixvGB91vnooJe9ytv...   
4  EUW1_7903341045  _e7dWtv-BUliXInflL39nVW5GhdZsLd61KeqTDK87OLfKs...   

  champion  team winning_team  kills  deaths  assists  gold_earned  \
0   Singed  Blue          Red      0       2        5         7742   
1    Elise  Blue          Red      2       7        7         8227   
2    Yasuo  Blue          Red      2       3        1         7868   
3    Varus  Blue          Red     10       7        1        12781   
4     Lulu  B

In [29]:
# Сохраняем общий датасет
final_df.to_csv("final_data.csv", index=False)
print("Сохранено: final_data.csv")

Сохранено: final_data.csv


Выясним корреляцию между длительностью матча и исходом.

In [36]:
# Средняя длительность для побед и поражений
avg_by_win = final_df.groupby("win")["game_duration_min"].mean()
print(avg_by_win)

win
False    27.654
True     27.654
Name: game_duration_min, dtype: float64


Как можно увидеть, длительность матча не влияет на победу.

### Финальное задание
1. Получить данные для другого игрока по его Riot ID (например, `Faker` / `KR`). Найти нужный endpoint в документации: https://developer.riotgames.com/apis#account-v1

2. Добавить в `matches_df` новую колонку `kda`:
   ```python
   kda = (kills + assists) / max(deaths, 1)
   ```

3. Посчитать средний KDA по каждому чемпиону и вывести топ-10.


Получим puuid по Riot ID

In [30]:
# Riot ID
GAME_NAME = "Faker"
TAG_LINE = "KR120"
REGION = "asia"

# Формируем URL
url = f"https://{REGION}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{GAME_NAME}/{TAG_LINE}"

response = requests.get(url, headers=headers)
print("Статус:", response.status_code)

if response.status_code == 200:
    account_data = response.json()
    puuid = account_data["puuid"]
    print(f"PUUID для {GAME_NAME}#{TAG_LINE}: {puuid}")
    
    # Если хотите, можно подставить этот puuid в запрос матчей (как в основном коде)
else:
    print("Ошибка:", response.status_code, response.text)

Статус: 200
PUUID для Faker#KR120: As0qynt_atHdRTbcHsaKi8cS5oy0K8j44ZlVOETf7HrBNpoErz-5za9U1KVgyztRPdndLdrB75VfKQ


Добавим колонки kda и расчёт среднего по чемпионам

In [31]:
# Добавляем колонку KDA
matches_df["kda"] = (matches_df["kills"] + matches_df["assists"]) / matches_df["deaths"].replace(0, 1)

# Проверим результат
matches_df.head()

,match_id,puuid,champion,team,winning_team,kills,deaths,assists,gold_earned,damage_to_champions,minions_killed,vision_score,win,role,team_position,game_duration_sec,game_duration_min,kda
0,EUW1_7903341045,4at6aYMfmRjhILQuNy_7vLOpKMWcl-8Sqq3qJ7lEC26OkS...,Singed,Blue,Red,0,2,5,7742,15291,204,18,False,NONE,TOP,1466,24.433333,2.500000
1,EUW1_7903341045,tN5_imed5Exq8ky2aOPLs9LOqDrWeM_sRwAelmFb19S03Z...,Elise,Blue,Red,2,7,7,8227,19059,6,22,False,NONE,JUNGLE,1466,24.433333,1.285714
2,EUW1_7903341045,1W-qsRJIfO3JVrKd4OyIjdcX4PUodkAsl8pXYhbfxOBaV-...,Yasuo,Blue,Red,2,3,1,7868,10858,171,14,False,SOLO,MIDDLE,1466,24.433333,1.000000
3,EUW1_7903341045,4gg2vM-4u8wfAwpw2ot3zDqsnRBrDixvGB91vnooJe9ytv...,Varus,Blue,Red,10,7,1,12781,22637,232,16,False,CARRY,BOTTOM,1466,24.433333,1.571429
4,EUW1_7903341045,_e7dWtv-BUliXInflL39nVW5GhdZsLd61KeqTDK87OLfKs...,Lulu,Blue,Red,1,2,13,7074,9932,26,103,False,SUPPORT,UTILITY,1466,24.433333,7.000000


Посчитаем средний KDA по каждому чемпиону

In [32]:
# Средний KDA по каждому чемпиону
top10_kda = (
    matches_df
    .groupby("champion")["kda"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

# Выведем топ-10 чемпионов по среднему KDA
print("Топ-10 чемпионов по среднему KDA:")
print(top10_kda)

Топ-10 чемпионов по среднему KDA:
champion
Rakan           21.000000
Orianna         16.000000
Shyvana         16.000000
Shaco           13.750000
Viktor          10.083333
Nidalee          9.466667
Akshan           9.166667
Vladimir         9.000000
Pyke             8.666667
FiddleSticks     8.000000
Name: kda, dtype: float64


Выгрузим полученные файлы также в SQL формате для дальнейшего анализа.

In [33]:
# Создаём подключение к БД
engine = create_engine("sqlite:///lol_data.db")

# Сохраняем таблицы в БД
try:
    players_df.to_sql("players", engine, if_exists="replace", index=False)
    print("Датасет players загружен")
except Exception as e:
    print(f"Ошибка при загрузке датасета players: {e}")

try:
    matches_df.to_sql("matches", engine, if_exists="replace", index=False)
    print("Датасет matches загружен")
except Exception as e:
    print(f"Ошибка при загрузке matches: {e}")

try:
    champions_df.to_sql("champions", engine, if_exists="replace", index=False)
    print("Датасет champions загружен")
except Exception as e:
    print(f"Ошибка при загрузке champions: {e}")

print("Все данные сохранены в БД!")

Датасет players загружен
Датасет matches загружен
Датасет champions загружен
Все данные сохранены в БД!
